# Quick Trainer on Google Colab

Fine-tune a Hugging Face language model with [Quick Trainer](https://github.com/MParvin/quick-trainer) using LoRA and optional 4-bit quantization, then upload the result to the Hugging Face Hub.

**Before you start:**
1. Go to **Runtime → Change runtime type → GPU** (T4 or better recommended).
2. Add a Colab secret named `HF_TOKEN` with your [Hugging Face write token](https://huggingface.co/settings/tokens).
3. Edit `huggingface.repo_id` in the config cell below to your target repo (e.g. `your-username/my-colab-model`).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable one via Runtime → Change runtime type → GPU, then re-run this cell."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q git+https://github.com/MParvin/quick-trainer.git@master

In [ ]:
import getpass
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (write access): ")
    print("HF_TOKEN set from prompt.")

In [ ]:
# Optional: mount Google Drive to persist outputs across sessions.
# Uncomment the lines below and set USE_DRIVE = True to save under Drive.

USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/quick-trainer/output"
else:
    OUTPUT_DIR = "/content/output"

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
from pathlib import Path

import yaml

CONFIG_PATH = Path("configs/colab.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

config = {
    "base_model": "meta-llama/Llama-3.2-1B-Instruct",
    "datasets": [
        {
            "path": "databricks/databricks-dolly-15k",
            "split": "train",
            "instruction_field": "instruction",
            "response_field": "response",
            "max_samples": 500,
        }
    ],
    "training": {
        "output_dir": OUTPUT_DIR,
        "num_train_epochs": 1,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2.0e-4,
        "max_seq_length": 512,
        "use_lora": True,
        "load_in_4bit": True,
        "bf16": True,
        "lora": {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05},
    },
    "huggingface": {
        "enabled": True,
        "repo_id": "your-username/my-colab-model",  # <-- edit this
        "private": False,
        "commit_message": "Fine-tuned with Quick Trainer on Colab",
    },
    "ollama": {"enabled": False},
}

CONFIG_PATH.write_text(yaml.dump(config, default_flow_style=False, sort_keys=False))
print(f"Wrote config to {CONFIG_PATH}")

In [ ]:
!quick-trainer validate {CONFIG_PATH}

In [ ]:
!quick-trainer train --config {CONFIG_PATH}

In [ ]:
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
export_dir = output_dir / "merged"

print(f"Training output: {output_dir}")
if export_dir.exists():
    print(f"Merged weights: {export_dir}")

repo_id = config["huggingface"]["repo_id"]
if config["huggingface"]["enabled"] and not repo_id.startswith("your-username/"):
    print(f"Hugging Face repo: https://huggingface.co/{repo_id}")